## Helmstetter HW 11: Dadi Lab

Load Packages

In [41]:
import msprime
import dadi
import numpy as np

Create object called demography

In [42]:
demography = msprime.Demography()

Create a population with N = 500 that originated from an ancestral population of Ne = 10,000 that bottlenecked 800 generations previously. Note that you have methods, or functions, that apply to objects (e.g., demography.add_population).

In [43]:
demography.add_population(name="pop",initial_size=500) # prsent size

# ancestral size before T=800 generations ago was 10,000
demography.add_population_parameters_change(
    time=800,initial_size=10_000, population="pop"
)

PopulationParametersChange(time=800, initial_size=10000, growth_rate=None, population='pop')

In [44]:
demography

Demography(populations=[Population(initial_size=500, growth_rate=0, name='pop', description='', extra_metadata={}, default_sampling_time=None, initially_active=None, id=0)], events=[PopulationParametersChange(time=800, initial_size=10000, growth_rate=None, population='pop')], migration_matrix=array([[0.]]))

Here we take 20 samples (40 diploid genomes) from our specified population (demography). For each sample, simulate 5 million base-pair sequence, with a recombination rate of 1*10^8-8 (per unit sequence, per unit time)

In [45]:
ts = msprime.sim_ancestry(
    samples={"pop": 20},
    demography=demography,
    sequence_length=5_000_000,
    recombination_rate=1e-8,
    random_seed=42,
)

Add mutations to simulated ancestry at a rate of 1*10^-8.

In [46]:
mts = msprime.sim_mutations(ts, rate=1e-8, random_seed=43)

In [47]:
%cd ~/bioe-591-genomics/students/Helmstetter/hw_output/11

/home/group/bioe-591-genomics/students/Helmstetter/hw_output/11


View and write simulated genotypes to a .vcf

In [48]:
mts.genotype_matrix()
with open("bottleneck_sim.vcf", "w") as f:
    mts.write_vcf(f)

Read in our .vcf and convert into a format dadi expects

In [49]:
vcf_file = "bottleneck_sim.vcf"
popfile = "pops.txt"
dd = dadi.Misc.make_data_dict_vcf(vcf_file, popfile)

Summarize data as site frequency spectrum (folded, as we do not know which allele is ancestral versus derived). fs is "projected" down to 30 chromosomes, a technique that averages the calculated frequency of the minor allele over all possible n = 30 subsamples to account for randomly distributed missing data. In short,we have 40 haplotypes (two per individual) each 5,000,000 base pairs long all representing the same genomic region across indivdiuals. So we have 40 gene copies called "chromosomes," we resample all possible ways to choose 30 gene copies out of those 40 and get frequencies for each SNP. I think. 

In [50]:
fs = dadi.Spectrum.from_data_dict(
    dd,
    pop_ids=["pop"],
    projections=[30],
    polarized=False, # folded SFS
)
print("Spectrum sample size:", fs.sample_sizes)
print("Segregating sites:", fs.S())

Spectrum sample size: [30]
Segregating sites: 2844.6406640560062


We now specify the models themselves. 

Developing an understanding of each line of code would require time and reading beyond our capacity. The important things to note are the structure of these function definitions (def name(arguments)), the fact that they return a site frequency spectrum object to compare to your empirical data, and that we are comparing a null model of a single population of constant size (snm) to a model of a population size change some time in the past (two_epoch). This second model has two free parameters: nu, which is the size of the population at present relative to the size of the ancestral population, and T, which is the time (in units of $2Ne_{ancestral}$ generations before present) of the population size change. The object $phi$ (or 
$phi(x)$) is the distribution of the number of polymorphic sites with a particular allele frequency; xx is the number of grid points used to approximate the curve of $phi$ (a larger value means greater accuracy but slower runtime).

In [51]:
def snm(params, ns, pts): # define single population model, no free parameters
    xx = dadi.Numerics.default_grid(pts)
    phi = dadi.PhiManip.phi_1D(xx)
    fs_model = dadi.Spectrum.from_phi(phi, ns, (xx,))
    return fs_model

def two_epoch(params, ns, pts): # define bottlenneck model
    nu, T = params # two free parameters: scaled current pop size and time of split 
    xx = dadi.Numerics.default_grid(pts)
    phi = dadi.PhiManip.phi_1D(xx)
    phi = dadi.Integration.one_pop(phi, xx, T, nu)
    fs_model = dadi.Spectrum.from_phi(phi, ns, (xx,))
    return fs_model

We calculate phi three times at increasing grid resolution, then use the differences among those calculations to extrapolate what phi would look like at infinite resolution. This creates new versions of the demographic models that perform this extrapolation automatically (snm_ex and two_ecpoch_ex).

In [52]:
pts_l = [40, 50, 60]
snm_ex = dadi.Numerics.make_extrap_log_func(snm)
two_epoch_ex = dadi.Numerics.make_extrap_log_func(two_epoch)

Fit the models to our data and compare the results. snm is the null model, it has no paramters ([]), feed it the sample size from out empirical SFS, and give it the grid point values ew input above. Results are the expected SFS under the constant-size model. We cale it to match the total diversity in our data by estimtaing the parameter $theta$ (=$4N_{e}\mu$). Compute log likelihood to measure how well the shape of the model SFS matches observed data.

In [53]:
model_snm = snm_ex([], fs.sample_sizes, pts_l)
theta_snm = dadi.Inference.optimal_sfs_scaling(model_snm, fs)
ll_snm = dadi.Inference.ll_multinom(model_snm, fs)

print("\nConstant-size model")
print("log-likelihood:", ll_snm)
print("theta:", theta_snm)


Constant-size model
log-likelihood: -430.50272646957603
theta: 719.989428931301


The process is more complex for our model of a population bottleneck, as we need to determine the best-fit parameter values before we calculate its log likelihood. To do this, we provide educated guesses for nu (here, 0.5, indicating a 50% contraction from its ancestral population size) and T (0.1 or $0.1 * 2N_{e}$ generations in the past), give these estiamtes lower and upper limits, and randomly perturb (randomly tweak original value a bit) the value within an order of magnitude of it's input value (fold=1). 

In [54]:
p0 = [0.5, 0.1]  
lower_bound = [1e-3, 1e-4]
upper_bound = [20, 10]
p0_perturbed = dadi.Misc.perturb_params(
    p0, fold=1, lower_bound=lower_bound, upper_bound=upper_bound
)

We then use an algorithm (of which there are several choices) to traverse the multinomial likelihood surface and (ideally) find its global maxima. The the dadi.Inference.optimize_log() function does this with these perturbed parameters, the empirical SFS, and the model in question; maxiter specifies the number of iterations to run.

In [55]:
popt = dadi.Inference.optimize_log(
    p0_perturbed,
    fs,
    two_epoch_ex,
    pts_l,
    lower_bound=lower_bound,
    upper_bound=upper_bound,
    verbose=1,
    maxiter=50,
)

180     , -158.946    , array([ 0.266892   ,  0.0502217  ])
181     , -159.111    , array([ 0.267159   ,  0.0502217  ])
182     , -158.863    , array([ 0.266892   ,  0.050272   ])
183     , -73.2099    , array([ 0.107955   ,  0.0786177  ])
184     , -73.2215    , array([ 0.108063   ,  0.0786177  ])
185     , -73.2131    , array([ 0.107955   ,  0.0786964  ])
189     , -73.2099    , array([ 0.107955   ,  0.0786177  ])
190     , -73.2215    , array([ 0.108063   ,  0.0786177  ])
191     , -73.2131    , array([ 0.107955   ,  0.0786963  ])
192     , -72.3715    , array([ 0.101255   ,  0.0701409  ])
193     , -72.3812    , array([ 0.101356   ,  0.0701409  ])
194     , -72.3712    , array([ 0.101255   ,  0.0702111  ])
195     , -71.6609    , array([ 0.0903651  ,  0.0634861  ])
196     , -71.6662    , array([ 0.0904555  ,  0.0634861  ])
197     , -71.6586    , array([ 0.0903651  ,  0.0635497  ])
198     , -71.3903    , array([ 0.0834164  ,  0.0622101  ])
199     , -71.3931    , array([ 0.083499

Fit the model with the optimized parameter values, scale to $theta$, and calculate likelihood.

In [56]:
model_two = two_epoch_ex(popt, fs.sample_sizes, pts_l)
theta_two = dadi.Inference.optimal_sfs_scaling(model_two, fs)
ll_two = dadi.Inference.ll_multinom(model_two, fs)

print("\nTwo-epoch model")
print("best params [nu, T]:", popt)
print("log-likelihood:", ll_two)
print("theta:", theta_two)


Two-epoch model
best params [nu, T]: [0.02278958 0.04998339]
log-likelihood: -70.74068496838953
theta: 7164.05087005773


Compare AIC to assess model fit.

In [57]:
print("\nModel comparison")
print(f"Delta log-likelihood (two-epoch - constant): {ll_two - ll_snm:.3f}")

# constant model has k=0 free params in this formulation
# two_epoch has k=2
aic_snm = 2 * 0 - 2 * ll_snm
aic_two = 2 * 2 - 2 * ll_two

print(f"AIC constant: {aic_snm:.3f}")
print(f"AIC two-epoch: {aic_two:.3f}")


Model comparison
Delta log-likelihood (two-epoch - constant): 359.762
AIC constant: 861.005
AIC two-epoch: 145.481


View the parameter values from the best-fit model. The "true" parameters are what we input into msprime (nu = 500/1000 = 0.5; T = 800/2 * 10000) = 0.04); the inferred parameters are stored in the optimized popt object.

In [58]:
print("True parameters:")
print("nu =", 0.05)
print("T  =", 0.04)

print("\nInferred parameters:")
print("nu =", popt[0])
print("T  =", popt[1])

True parameters:
nu = 0.05
T  = 0.04

Inferred parameters:
nu = 0.02278958380497424
T  = 0.049983392104640326


Scale into "real" units

In [59]:
N_anc = 10_000  # known from simulation
nu_est, T_est = popt # label popt parameter estimates
N_curr_est = nu_est * N_anc # multiply Nu in dadi units by ancestral populaiton size to get current N_e
t_est = T_est * 2 * N_anc # multiply T in dadi units by 2N_e to get generations
print("Estimated current size:", N_curr_est)
print("Estimated bottleneck time (generations):", t_est)

Estimated current size: 227.8958380497424
Estimated bottleneck time (generations): 999.6678420928065


### Discussion
The results suggest that the two epoch model fit the data better than the constant population size model (AIC = 145.482, 861.005 respectively). The true population size was 500 individuals and experienced a bottle neck 800 generations ago. The results did not accurately recover the simulated parameters, estimating a current population size of 264 individuals and a bottleneck occurring roughly 1,077 generations ago. While dadi identified a bottleneck, the timing (~277 generations off) would be relatively minor in similar contexts. However, underestimating the current population size by roughly half could have important implications for species under management.